# מפגש 13 — מחברת מורה מודרכת
היום משווים שני סרגלים ונמנעים מדליפה. **לא מאמנים מודל ולא מחשבים מדדי חיזוי.** המחברת מממשת דוגמה שונה מן המעבדה העצמאית.

## פתיחה: מחיר ו־RAM אינם אותה יחידה
מרחק נאיבי בין מחשבים [2000₪,8GB] ו־[2100₪,16GB] הוא √(100²+8²)≈100.32; זהו ערבוב יחידות ולא מרחק פיזיקלי. שינוי קנה מידה הוא הכנה אפשרית למרחקים; **אינו** הוכחה ששני מאפיינים חשובים באותה מידה.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler,MinMaxScaler
raw_naive_distance=np.sqrt((2100-2000)**2+(16-8)**2)
print('Naive numeric distance',raw_naive_distance)
assert np.isclose(raw_naive_distance,np.sqrt(10064))

## דוגמה ללוח: Train=[10,20,30], Test=[40]
חישוב ביד תחילה: min10,max30,mean20,σ=√(200/3)≈8.165. MinMax Test 40 נותן 1.5; Standard Test 40 נותן 2.449.

In [ ]:
train=pd.DataFrame({'measurement':[10.,20.,30.]})
test=pd.DataFrame({'measurement':[40.]})
mean_train=train['measurement'].mean()
std_train=train['measurement'].std(ddof=0)
min_train=train['measurement'].min()
max_train=train['measurement'].max()
print('mean,std,min,max',mean_train,std_train,min_train,max_train)
assert np.isclose(mean_train,20)
assert np.isclose(std_train,np.sqrt(200/3))
assert (min_train,max_train)==(10.,30.)

## נוסחאות ידניות ובדיקת תוצאות
עבור x מחשבים (x−min_train)/(max_train−min_train) או (x−mean_train)/std_train. הנתונים החדשים אינם מלמדים פרמטרים.

In [ ]:
manual_minmax=(test['measurement']-min_train)/(max_train-min_train)
manual_z=(test['measurement']-mean_train)/std_train
print('Manual Test:',manual_minmax.to_numpy(),manual_z.to_numpy())
assert np.allclose(manual_minmax,[1.5])
assert np.allclose(manual_z,[np.sqrt(6)])

## בדיקה בספרייה: Fit רק על Train
set_output משאיר שמות עמודות כדי שנוכל להשוות פלטים של DataFrame.

In [ ]:
mm=MinMaxScaler().set_output(transform='pandas')
st=StandardScaler().set_output(transform='pandas')
mm.fit(train)
st.fit(train)
mm_train=mm.transform(train)
mm_test=mm.transform(test)
st_train=st.transform(train)
st_test=st.transform(test)
display(mm_train,mm_test,st_train,st_test)
assert np.allclose(mm_train['measurement'],[0.,0.5,1.])
assert np.allclose(mm_test['measurement'],[1.5])
assert np.allclose(st_train['measurement'],[-np.sqrt(1.5),0.,np.sqrt(1.5)])
assert np.allclose(st_test['measurement'],[np.sqrt(6)])

## בדיקת מאפייני ההתאמה
MinMax לומד גבולות, Standard לומד ממוצע וסטיית תקן עם ddof=0. תקנון Train מביא ממוצע 0 וסטיית תקן 1, אבל לא הופך את הצורה לנורמלית.

In [ ]:
print('MinMax boundaries:',mm.data_min_,mm.data_max_)
print('Standard mean and scale:',st.mean_,st.scale_)
assert np.allclose(mm.data_min_,[10.]) and np.allclose(mm.data_max_,[30.])
assert np.allclose(st.mean_,[20.])
assert np.allclose(st.scale_,[np.sqrt(200/3)])
assert np.isclose(st_train['measurement'].mean(),0.)
assert np.isclose(st_train['measurement'].std(ddof=0),1.)
assert not np.isclose(st_test['measurement'].mean(),0.)

## ציור נקודות על שני סרגלים
התבוננו ב־Test=40: 1.5 הוא מעבר למקסימום האימון בסרגל MinMax ואינו שגיאה.

In [ ]:
import matplotlib.pyplot as plt
fig,axs=plt.subplots(1,2,figsize=(10,3.5))
axs[0].scatter([10,20,30],[1,1,1],label='Train',s=60)
axs[0].scatter([40],[1],label='Test',marker='x',s=100)
axs[0].set_xlabel('Original measurement')
axs[0].set_yticks([])
axs[0].legend()
axs[1].scatter(mm_train['measurement'],[1,1,1],label='Train',s=60)
axs[1].scatter(mm_test['measurement'],[1],label='Test',marker='x',s=100)
axs[1].set_xlabel('MinMax scaled (fit on Train)')
axs[1].set_yticks([])
axs[1].legend()
plt.tight_layout()
plt.show()

## הרחבה למורה: עמודה קבועה
ידנית אין לחלק ב־0. ספריית Scikit-learn משתמשת בטיפול בטוח. זו הרחבה שאינה חלק ממעבדת התלמיד.

In [ ]:
constant_train=np.array([[7.],[7.],[7.]])
mm_constant=MinMaxScaler().fit(constant_train)
st_constant=StandardScaler().fit(constant_train)
assert np.allclose(mm_constant.transform(constant_train),0.)
assert np.allclose(st_constant.transform(constant_train),0.)
assert np.allclose(mm_constant.transform([[9.]]),[[2.]])
assert np.allclose(st_constant.transform([[9.]]),[[2.]])
print('Constant-column Train maps to 0; Test=9 maps to 2 in the current library')

## שאלות יציאה
האם Test חייבת להיות ב־[0,1]? לא. האם ממוצע Test בתקנון חייב להיות 0? לא. האם Standard יוצר נורמליות? לא. האם Fit חדש על Test תקין? לא.